# CLIP's contrastive loss, from scratch
**LLMs From Basics · V1.2 hands-on** &nbsp; [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhishekaryan23/LLM-Learner/blob/main/course/notebooks/v1-2-clip-loss.ipynb)

In the lesson, CLIP turned a batch of image–caption pairs into an $N \times N$ similarity matrix and scored it with a symmetric cross-entropy. Here you'll build that loss and use it:

1. **normalise** embeddings and compute the **logit matrix** $S/\tau$
2. the **symmetric InfoNCE loss** in NumPy, checked against PyTorch's `cross_entropy`
3. **train two tiny linear "encoders"** so paired toy "image" and "text" features land close together
4. **zero-shot classification**: label held-out "images" by their nearest class "prompt"

No real images: the "images" and "captions" are toy feature vectors generated from shared hidden concepts, so everything runs in seconds.

**How to use it:** run cells top to bottom (Shift+Enter). Each `# TODO` cell is followed by a ✅ check cell. Stuck for more than five minutes? The **Solutions** are at the bottom.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)
torch.manual_seed(0)

# Toy world: 5 concepts ("dog", "cat", ...). Each pair shares a hidden vector z near its concept.
# The "image" sees z through one nonlinear map, the "caption" through another, plus noise.
K, DZ, DI, DT = 5, 6, 20, 12
CLASSES = ["dog", "cat", "car", "tree", "boat"]
centers = rng.normal(size=(K, DZ)) * 2.0
A_img = rng.normal(size=(DZ, DI))
B_txt = rng.normal(size=(DZ, DT))

def make_pairs(n, rng):
    y = rng.integers(0, K, n)
    z = centers[y] + 0.6 * rng.normal(size=(n, DZ))
    X = np.tanh(z @ A_img / 3) + 0.05 * rng.normal(size=(n, DI))   # "image" features
    T = z @ B_txt + 0.05 * rng.normal(size=(n, DT))                 # "caption" features
    return X.astype(np.float32), T.astype(np.float32), y

X_train, T_train, y_train = make_pairs(4000, rng)
X_test, T_test, y_test = make_pairs(1000, rng)
# A class "prompt" is the caption feature of the concept centre ("a photo of a dog").
prompts = (centers @ B_txt).astype(np.float32)
print("image features", X_train.shape, " caption features", T_train.shape, " prompts", prompts.shape)

## 1. Unit vectors and the logit matrix
CLIP compares embeddings by **cosine similarity**: scale each row to length 1, then take dot products. Write

* `l2_normalize(E)`: divide each row of `E` by its Euclidean norm,
* `logit_matrix(img, txt, tau)`: normalise both, return the $N \times N$ matrix with entry $(i, j) = \cos(\text{img}_i, \text{txt}_j) / \tau$.

In [ ]:
def l2_normalize(E):
    # TODO
    raise NotImplementedError("Fill in l2_normalize")

def logit_matrix(img, txt, tau):
    # TODO
    raise NotImplementedError("Fill in logit_matrix")

In [ ]:
E = rng.normal(size=(6, 4))
assert np.allclose(np.linalg.norm(l2_normalize(E), axis=1), 1.0), "Every row should have length 1."
a, b = rng.normal(size=(3, 5)), rng.normal(size=(3, 5))
L = logit_matrix(a, b, 0.1)
assert L.shape == (3, 3)
cos01 = a[0] @ b[1] / np.linalg.norm(a[0]) / np.linalg.norm(b[1])
assert np.isclose(L[0, 1], cos01 / 0.1), "Entry (0, 1) should be cos(img_0, txt_1) / tau."
assert np.all(np.abs(L) <= 1 / 0.1 + 1e-9), "Cosine similarities are between -1 and 1."
print("✅ logit matrix works:\n", L)

## 2. The symmetric InfoNCE loss
$$\mathcal{L} = \frac{1}{2N}\sum_{i=1}^{N}\left[-\log\frac{e^{\ell_{ii}}}{\sum_{j} e^{\ell_{ij}}} - \log\frac{e^{\ell_{ii}}}{\sum_{j} e^{\ell_{ji}}}\right], \qquad \ell = S/\tau$$

Implement `clip_loss(img, txt, tau)` in NumPy. Use a numerically stable log-softmax: subtract each row's (or column's) maximum before exponentiating. The right answer for row $i$ and column $i$ is always the diagonal.

In [ ]:
def log_softmax(x, axis):
    # given: stable log-softmax along an axis
    m = x.max(axis=axis, keepdims=True)
    return x - m - np.log(np.exp(x - m).sum(axis=axis, keepdims=True))

def clip_loss(img, txt, tau):
    # TODO: logits -> row-wise and column-wise cross-entropy with the diagonal as target -> average
    raise NotImplementedError("Fill in clip_loss")

In [ ]:
# Reference: the standard PyTorch implementation
def clip_loss_torch(img, txt, tau):
    i = F.normalize(torch.tensor(img, dtype=torch.float64), dim=1)
    t = F.normalize(torch.tensor(txt, dtype=torch.float64), dim=1)
    logits = i @ t.T / tau
    labels = torch.arange(len(img))
    return ((F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2).item()

for n, d, tau in [(4, 8, 0.07), (32, 16, 0.5), (128, 8, 0.01)]:
    a, b = rng.normal(size=(n, d)), rng.normal(size=(n, d))
    assert np.isclose(clip_loss(a, b, tau), clip_loss_torch(a, b, tau)), (n, d, tau)
# sanity checks: perfect alignment with a low temperature -> ~0; random at tau = 1 -> about log N
same = rng.normal(size=(16, 32))
assert clip_loss(same, same, 0.01) < 1e-3
print(f"random, tau=1, N=64: loss {clip_loss(rng.normal(size=(64, 32)), rng.normal(size=(64, 32)), 1.0):.3f}  vs  log 64 = {np.log(64):.3f}")
print("✅ clip_loss matches PyTorch's cross_entropy in both directions.")

## 3. Train two tiny encoders
The "image encoder" is a linear map $\mathbb{R}^{20} \to \mathbb{R}^{8}$ and the "text encoder" a linear map $\mathbb{R}^{12} \to \mathbb{R}^{8}$. The temperature is learned too, stored as `log_scale` $= \log(1/\tau)$ (CLIP starts at $\tau = 0.07$ and caps the scale at 100).

Fill in `clip_step`: embed the batch with both encoders, normalise (`F.normalize`), build `logits = scale * img @ txt.T`, and return the symmetric cross-entropy (use `F.cross_entropy`, so autograd can do the rest).

In [ ]:
img_enc = torch.nn.Linear(DI, 8)
txt_enc = torch.nn.Linear(DT, 8)
log_scale = torch.nn.Parameter(torch.tensor(np.log(1 / 0.07), dtype=torch.float32))

def clip_step(xb, tb):
    scale = log_scale.exp().clamp(max=100)
    # TODO: return the symmetric CLIP loss for this batch (a scalar tensor)
    raise NotImplementedError("Fill in clip_step")

In [ ]:
def retrieval_top1(X, T):
    # fraction of images whose most similar caption (in the same batch) is their own
    with torch.no_grad():
        a = F.normalize(img_enc(torch.tensor(X)), dim=1)
        b = F.normalize(txt_enc(torch.tensor(T)), dim=1)
        return ((a @ b.T).argmax(1) == torch.arange(len(X))).float().mean().item()

before = retrieval_top1(X_test[:256], T_test[:256])
opt = torch.optim.Adam(list(img_enc.parameters()) + list(txt_enc.parameters()) + [log_scale], lr=1e-2)
Xt, Tt, losses = torch.tensor(X_train), torch.tensor(T_train), []
for step in range(600):
    idx = torch.randint(0, len(Xt), (256,))
    loss = clip_step(Xt[idx], Tt[idx])
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
after = retrieval_top1(X_test[:256], T_test[:256])
print(f"loss {losses[0]:.2f} -> {np.mean(losses[-50:]):.3f}; learned tau = {1 / log_scale.exp().clamp(max=100).item():.4f} (0.01 is the cap, which CLIP also reached)")
print(f"held-out retrieval top-1 among 256 pairs: {before:.3f} before training -> {after:.3f} after")
assert np.mean(losses[-50:]) < 1.0 and after > 0.8, "Training should align the pairs. Check clip_step."
plt.figure(figsize=(5, 2.6)); plt.plot(losses); plt.xlabel("step"); plt.ylabel("CLIP loss"); plt.yscale("log"); plt.show()
print("✅ The two encoders now map paired features close together.")

## 4. Zero-shot classification
Embed the 5 class prompts with the **text** encoder and the held-out "images" with the **image** encoder, normalise both, and predict the class whose prompt has the highest cosine similarity. The encoders never saw a class label.

In [ ]:
def zero_shot_predict(X, prompts):
    # TODO: return an integer array of predicted class indices, one per row of X
    raise NotImplementedError("Fill in zero_shot_predict")

In [ ]:
pred = zero_shot_predict(X_test, prompts)
acc = (np.asarray(pred) == y_test).mean()
print(f"zero-shot accuracy on {len(y_test)} held-out images: {acc:.3f} (chance = {1 / K:.2f})")
assert acc > 0.85, "Expected well above chance. Check zero_shot_predict."
print("✅ Zero-shot works: classes defined only by their text prompts.")

## Experiment
1. **Freeze the temperature** at $\tau = 1$ (replace `log_scale.exp()` with `torch.tensor(1.0)`) and retrain. What happens to the loss and to retrieval? Then try $\tau = 0.01$ from the start.
2. **Batch size.** Retrain with batches of 16 and 1024. Which gives better retrieval for the same number of steps, and why (think: negatives)?
3. **SigLIP.** The cell below implements the sigmoid loss $-\frac{1}{N}\sum_{i,j}\log\sigma\big(z_{ij}(t\,s_{ij}+b)\big)$ with $z_{ij} = +1$ on the diagonal and $-1$ elsewhere. Swap it into `clip_step` (it needs a learned bias $b$, initialised at $-10$, and $t$ initialised at 10) and compare.

In [ ]:
def siglip_loss(img_emb, txt_emb, t, b):
    # img_emb, txt_emb: (N, d) unit vectors; t, b: scalar tensors
    logits = t * img_emb @ txt_emb.T + b
    z = 2 * torch.eye(len(img_emb)) - 1                      # +1 on the diagonal, -1 elsewhere
    return -F.logsigmoid(z * logits).sum() / len(img_emb)

# quick check on random unit vectors
a = F.normalize(torch.randn(8, 4), dim=1); b_ = F.normalize(torch.randn(8, 4), dim=1)
print("SigLIP loss on random vectors:", siglip_loss(a, b_, torch.tensor(10.0), torch.tensor(-10.0)).item())

---
## Solutions
Try the exercises first. Each solution below can be pasted into its TODO cell.

In [ ]:
# Solution: unit vectors and the logit matrix
def l2_normalize(E):
    return E / np.linalg.norm(E, axis=1, keepdims=True)

def logit_matrix(img, txt, tau):
    return l2_normalize(img) @ l2_normalize(txt).T / tau

In [ ]:
# Solution: symmetric InfoNCE
def log_softmax(x, axis):
    m = x.max(axis=axis, keepdims=True)
    return x - m - np.log(np.exp(x - m).sum(axis=axis, keepdims=True))

def clip_loss(img, txt, tau):
    L = logit_matrix(img, txt, tau)
    diag = np.arange(len(L))
    loss_img = -log_softmax(L, axis=1)[diag, diag].mean()   # each image picks its caption (rows)
    loss_txt = -log_softmax(L, axis=0)[diag, diag].mean()   # each caption picks its image (columns)
    return (loss_img + loss_txt) / 2

In [ ]:
# Solution: one training step's loss
img_enc = torch.nn.Linear(DI, 8)
txt_enc = torch.nn.Linear(DT, 8)
log_scale = torch.nn.Parameter(torch.tensor(np.log(1 / 0.07), dtype=torch.float32))

def clip_step(xb, tb):
    scale = log_scale.exp().clamp(max=100)
    a = F.normalize(img_enc(xb), dim=1)
    b = F.normalize(txt_enc(tb), dim=1)
    logits = scale * a @ b.T
    labels = torch.arange(len(xb))
    return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2

In [ ]:
# Solution: zero-shot classification
def zero_shot_predict(X, prompts):
    with torch.no_grad():
        a = F.normalize(img_enc(torch.tensor(X)), dim=1)
        p = F.normalize(txt_enc(torch.tensor(prompts)), dim=1)
        return (a @ p.T).argmax(1).numpy()